# Single Reciter End-to-End Forced Alignment Pipeline (Colab GPU)

High-throughput, overlapped asynchronous forced alignment and audio normalization for a single reciter (114 surahs).

**Performance Config:** `--device cuda --cuda-batch-size 8 --intra-surah-split --prefetch-workers 4 --prefetch-batches 2`
**Output:** EBU R128 Loudnorm Opus audio + word-level timed JSON & SRT → Google Drive

In [ ]:
# 1. Install package + CUDA extra & dependencies
!pip install -q git+https://github.com/HsnSaboor/quran-forced-align.git --extra cuda 2>/dev/null || \
!pip install -q -e . --extra cuda
!pip install -q curl_cffi tqdm huggingface_hub
!apt-get install -y -qq ffmpeg > /dev/null 2>&1
print("✓ Environment and CUDA dependencies installed.")

In [ ]:
# 2. Mount Google Drive & configure storage layout
import os
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT   = '/content/drive/MyDrive/quran_forced_align'
DRIVE_MODEL  = f'{DRIVE_ROOT}/model'
DRIVE_OPUS   = f'{DRIVE_ROOT}/opus'
DRIVE_JSON   = f'{DRIVE_ROOT}/json'
DRIVE_AUDIO  = f'{DRIVE_ROOT}/audio_input'

# Local high-speed NVMe scratch cache
LOCAL_AUDIO  = '/content/audio_cache'
LOCAL_OUTPUT = '/content/output_cache'
LOCAL_MODEL  = '/content/model'

for d in [DRIVE_ROOT, DRIVE_MODEL, DRIVE_OPUS, DRIVE_JSON, DRIVE_AUDIO, LOCAL_AUDIO, LOCAL_OUTPUT, LOCAL_MODEL]:
    os.makedirs(d, exist_ok=True)

print(f"✓ Drive mounted. Output root: {DRIVE_ROOT}")

In [ ]:
# 3. Automated Model Download: Quran-Lab/zipformer_p-arabic-v3 + Drive Cache
import os, shutil
from huggingface_hub import hf_hub_download

MODEL_REPO = "Quran-Lab/zipformer_p-arabic-v3"
MODEL_FILE = "zipformer_p_arabic_v3.int8.onnx"
TOKENS_FILE = "tokens.txt"

local_model_path = os.path.join(LOCAL_MODEL, MODEL_FILE)
local_tokens_path = os.path.join(LOCAL_MODEL, TOKENS_FILE)
drive_model_path = os.path.join(DRIVE_MODEL, MODEL_FILE)
drive_tokens_path = os.path.join(DRIVE_MODEL, TOKENS_FILE)

# Check Colab Secrets or Env for HF token
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')

# Restore from Drive or download from HF
if os.path.exists(drive_model_path) and os.path.exists(drive_tokens_path):
    print(f"✓ Restoring model from Google Drive cache: {drive_model_path}")
    shutil.copy2(drive_model_path, local_model_path)
    shutil.copy2(drive_tokens_path, local_tokens_path)
else:
    print(f"Downloading {MODEL_FILE} from Hugging Face ({MODEL_REPO})...")
    dl_model = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE, token=hf_token)
    dl_tokens = hf_hub_download(repo_id=MODEL_REPO, filename=TOKENS_FILE, token=hf_token)
    shutil.copy2(dl_model, local_model_path)
    shutil.copy2(dl_tokens, local_tokens_path)
    shutil.copy2(dl_model, drive_model_path)
    shutil.copy2(dl_tokens, drive_tokens_path)
    print(f"✓ Model cached to Google Drive: {DRIVE_MODEL}")

print(f"✓ Model ready: {local_model_path} ({os.path.getsize(local_model_path)/1024/1024:.1f} MB)")

In [ ]:
# 4. Select Reciter
RECITER = {
    'name': 'mishary-rashid-alafasy',
    'slug': 'alafasy',
    'reciter_id': 1,
    'collection_id': 1,
}

REC_SLUG = RECITER['slug']
LOCAL_REC_AUDIO = f"{LOCAL_AUDIO}/{REC_SLUG}"
DRIVE_REC_JSON = f"{DRIVE_JSON}/{REC_SLUG}"
DRIVE_REC_OPUS = f"{DRIVE_OPUS}/{REC_SLUG}"

os.makedirs(LOCAL_REC_AUDIO, exist_ok=True)
os.makedirs(DRIVE_REC_JSON, exist_ok=True)
os.makedirs(DRIVE_REC_OPUS, exist_ok=True)

print(f"✓ Target reciter: {RECITER['name']} ({REC_SLUG})")
print(f"  Local Audio Cache: {LOCAL_REC_AUDIO}")
print(f"  Drive JSON Output: {DRIVE_REC_JSON}")
print(f"  Drive Opus Output: {DRIVE_REC_OPUS}")

In [ ]:
# 5. Async Audio Prefetcher (curl_cffi)
import asyncio, glob
from curl_cffi.requests import AsyncSession

async def download_all_surahs_async(reciter: dict, out_dir: str, max_concurrent: int = 10) -> int:
    rec_id = reciter["reciter_id"]
    coll_id = reciter.get("collection_id", 1)
    existing = {int(os.path.basename(f).split(".")[0]) for f in glob.glob(f"{out_dir}/*.mp3")}
    if len(existing) == 114:
        print("✓ All 114 surah audio files already present locally.")
        return 114

    print(f"Fetching recitation list for reciter {rec_id}...")
    async with AsyncSession(timeout=15, impersonate="chrome120") as session:
        resp = await session.get(f"https://www.assabile.com/ajax/loadplayer-{rec_id}-{coll_id}")
        if resp.status_code != 200:
            raise RuntimeError(f"Failed to query assabile loadplayer: status {resp.status_code}")
        
        recs = resp.json().get("Recitation", [])
        print(f"Found {len(recs)} available surah recitations. Downloading concurrently...")
        
        sem = asyncio.Semaphore(max_concurrent)
        async def fetch_one(r):
            s_id = r.get("sura_id")
            href = r.get("href", "").lstrip("#")
            if not s_id or not href:
                return False
            s_num = int(s_id)
            target = os.path.join(out_dir, f"{s_num:03d}.mp3")
            if os.path.exists(target) and os.path.getsize(target) > 1000:
                return True
            
            async with sem:
                for attempt in range(3):
                    try:
                        r_link = await session.get(f"https://www.assabile.com/ajax/getrcita-link-{href}", timeout=10)
                        if r_link.status_code == 200 and r_link.text.strip().startswith("http"):
                            mp3_url = r_link.text.strip()
                            r_audio = await session.get(mp3_url, timeout=45)
                            if r_audio.status_code == 200 and len(r_audio.content) > 1000:
                                with open(target, "wb") as f:
                                    f.write(r_audio.content)
                                return True
                    except Exception:
                        await asyncio.sleep(1.0 * (attempt + 1))
            return False

        tasks = [fetch_one(r) for r in recs]
        results = await asyncio.gather(*tasks)
        return sum(1 for res in results if res)

count = asyncio.run(download_all_surahs_async(RECITER, LOCAL_REC_AUDIO))
print(f"✓ Download complete: {count}/114 audio files available in {LOCAL_REC_AUDIO}")

In [ ]:
# 6. High-Throughput Overlapped Forced Alignment & Transcoding
import time, glob
from quran_forced_align.batch_cli import run_pipelined_batch

available = sorted([int(os.path.basename(f).split(".")[0]) for f in glob.glob(f"{LOCAL_REC_AUDIO}/*.mp3")])
if not available:
    raise FileNotFoundError(f"No audio files found in {LOCAL_REC_AUDIO}")

print(f"Aligning surahs {min(available)}-{max(available)} ({len(available)} surahs)...")
print(f"Config: --device cuda --cuda-batch-size 8 --intra-surah-split --prefetch-workers 4 --prefetch-batches 2")

t0 = time.monotonic()
summary = run_pipelined_batch(
    surah_list=available,
    audio_dir=LOCAL_REC_AUDIO,
    out_dir=DRIVE_REC_JSON,
    opus_dir=DRIVE_REC_OPUS,
    transcode_opus=True,
    model_path=local_model_path,
    tokens_path=local_tokens_path,
    device="cuda",
    cuda_batch_size=8,
    intra_surah_split=True,
    prefetch_workers=4,
    prefetch_batches=2,
    verbose=True,
)

elapsed = time.monotonic() - t0
print(f"\n✓ Forced alignment complete in {elapsed/60:.2f} mins!")

In [ ]:
# 7. Verification & Summary Report
import json

json_files = sorted(glob.glob(f"{DRIVE_REC_JSON}/*.json"))
opus_files = sorted(glob.glob(f"{DRIVE_REC_OPUS}/*.opus"))

print("=" * 70)
print(f"RECITER: {RECITER['name']} ({REC_SLUG})")
print(f"Surahs Processed: {summary['succeeded_count']}/{len(available)} (Failed: {summary['failed_count']})")
print(f"JSON Files Generated: {len(json_files)} → {DRIVE_REC_JSON}")
print(f"Opus Files Generated: {len(opus_files)} → {DRIVE_REC_OPUS}")
print(f"Total Words Aligned: {summary['total_words']:,}")
print(f"Repeats Detected: {summary['total_repeats']:,}")
print(f"Audio Hours: {summary['total_audio_sec']/3600:.2f}h | Wall-clock: {summary['total_wall_sec']/60:.2f}m")
print(f"Overall RTF: {summary['overall_rtf']:.4f}x ({1.0/max(summary['overall_rtf'], 1e-6):.1f}x realtime)")
print("=" * 70)

# Sample inspection of Surah Al-Fatiha
if json_files:
    with open(json_files[0], 'r', encoding='utf-8') as f:
        sample_records = json.load(f)
    first_word = sample_records[0]
    print(f"\nSample Record (Surah {first_word.get('sura', 1)}): {len(sample_records)} words total")
    print(f"First word: '{first_word.get('word')}' [{first_word.get('start'):.3f}s -> {first_word.get('end'):.3f}s]")
    print(f"Letter tier elements: {len(first_word.get('letters', []))}")